# Keystroke Dynamics Continuous Authentication Pipeline
### End-to-End Behavioral Biometric Verification under Zero Trust Architecture

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsrashedsatter/thesis-on-keystroke-dynamics-authentication/blob/main/keystroke_dynamics_pipeline.ipynb)

---

## 📌 Project & Pipeline Overview
This notebook provides the **complete, end-to-end research pipeline** for behavior-based user authentication using keystroke dynamics:

```
  [ Stage 01: Raw Ingestion ] ──────► 42,939 Keystrokes across 60 Typists (15 sessions/user)
              │
              ▼
  [ Stage 02: Cleaning & EDA ] ─────► Jitter Removal, Rollover Flagging, 99.98% Clean Retention
              │
              ▼
  [ Stage 03: Feature Engineering ] ► 124 Biometrics (Global Stats + Top 25 Digraph Latencies)
              │
              ▼
  [ Stage 04: Continuous Models ] ──► Random Forest (100%), BiLSTM (87.8%), 1D-CNN (52.8% - 65.6%)
```

- **Cohort**: 60 verified participants with 15 sentence sessions each (**900 sessions** total).
- **Dataset**: Aalto University 136 Million Keystrokes Benchmark Corpus.
- **Task**: Closed-Set Continuous Multiclass Identification across 60 typists (Random Chance = $1/60 \approx 1.67\%$).

## 1. ⚙️ Hardware Diagnostics & Environment Setup

In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
print(f"Python Version: {sys.version.split()[0]}")
print(f"Running inside Google Colab: {IN_COLAB}")

# Check for GPU acceleration
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    print(f"TensorFlow Version: {tf.__version__}")
    if gpus:
        print(f"✅ GPU Detected: {gpus}")
    else:
        print("ℹ️ No GPU detected. Running on CPU (CPU is completely fine for this feature matrix).")
        print("   Tip for Colab: Runtime -> Change runtime type -> Select 'T4 GPU' for faster training.")
except ImportError:
    print("TensorFlow will be installed in the next cell.")

## 2. 📥 Clone Repository & Install Dependencies

In [ ]:
if IN_COLAB:
    # Clone repository if not already present
    if not os.path.exists("thesis-on-keystroke-dynamics-authentication"):
        !git clone https://github.com/itsrashedsatter/thesis-on-keystroke-dynamics-authentication.git
        %cd thesis-on-keystroke-dynamics-authentication
    else:
        %cd thesis-on-keystroke-dynamics-authentication
        !git pull origin main

# Install dependencies efficiently
!pip install -q scikit-learn pandas numpy matplotlib seaborn scipy joblib pyarrow tensorflow
print("✅ Dependencies successfully installed and verified.")

## 3. 🔬 Stage 01 — Raw Telemetry Ingestion & Cohort Verification
Inspect the merged raw keystroke telemetry ingested from the Aalto 136 Million Keystroke dataset across the selected 60 typists.

In [ ]:
import pandas as pd
import numpy as np

raw_path = "01_data_merge/merged_raw_keystrokes.csv"
df_raw = pd.read_csv(raw_path, encoding='latin-1')

print(f"==================================================")
print(f"📊 STAGE 01 RAW DATASET SUMMARY:")
print(f"   - Total Raw Keystroke Events: {len(df_raw):,}")
print(f"   - Unique Participants:        {df_raw['PARTICIPANT_ID'].nunique()}")
print(f"   - Sessions per Participant:   {df_raw.groupby('PARTICIPANT_ID')['TEST_SECTION_ID'].nunique().iloc[0]} sessions each")
print(f"   - Total Unique Sessions:      {df_raw.groupby(['PARTICIPANT_ID', 'TEST_SECTION_ID']).ngroups}")
print(f"==================================================")

df_raw[['PARTICIPANT_ID', 'TEST_SECTION_ID', 'PRESS_TIME', 'RELEASE_TIME', 'LETTER', 'KEYCODE', 'AVG_WPM_15']].head(5)

## 4. 🧹 Stage 02 — Data Cleaning & Exploratory Data Analysis (EDA)
Execute `clean_and_eda.py` to:
1. Calculate Hold Time ($HT = \text{Release} - \text{Press}$), Flight Time ($UD$), and Down-to-Down Latency ($DD$).
2. Filter hardware jitter ($HT \le 0$) and extreme disconnect artifacts ($HT > 3000$ ms) with **99.98% retention**.
3. Flag key rollover events ($UD < 0$) and cognitive pauses ($DD > 5000$ ms).
4. Generate high-resolution biometric distribution charts.

In [ ]:
# Execute Data Cleaning and EDA Pipeline
!python clean_and_eda.py

In [ ]:
from IPython.display import Image, display

print("📈 Displaying Stage 02 Exploratory Visualizations:")

# 1. Overall Timing Distributions
print("\n1. Overall Timing Distributions (Hold Time, Flight Time, Down-to-Down Latency):")
display(Image("02_cleaning_eda/plots/overall_timing_distributions.png", width=800))

# 2. Inter-Participant Overlay Distributions
print("\n2. Inter-Participant Overlay (Demonstrating Individual Typist Differences):")
display(Image("02_cleaning_eda/plots/participant_overlay_distributions.png", width=800))

# 3. Timing Correlation Heatmap
print("\n3. Feature Correlation Heatmap:")
display(Image("02_cleaning_eda/plots/timing_correlation_heatmap.png", width=600))

# 4. Session Balance per Participant
print("\n4. Participant Session Balance:")
display(Image("02_cleaning_eda/plots/participant_session_balance.png", width=800))

## 5. 🛠️ Stage 03 — Feature Engineering & Preprocessing
Execute `feature_engineering_and_preprocessing.py` to:
1. Extract the **Top 25 Digraphs** across the entire experimental cohort.
2. Build **124 Biometric Features** per typing session (12 global stats, 6 cadence metrics, 5 error metrics, 100 digraph latencies).
3. Apply two-tier hierarchical imputation for absent digraphs.
4. Partition into **80/20 Stratified Train/Test splits** with zero data leakage (`StandardScaler` fitted strictly on training partition).

In [ ]:
# Execute Feature Engineering Pipeline
!python feature_engineering_and_preprocessing.py

In [ ]:
df_train = pd.read_csv("03_feature_engineering_preprocessing/train_scaled.csv")
df_test = pd.read_csv("03_feature_engineering_preprocessing/test_scaled.csv")

print("==================================================")
print("📊 STAGE 03 FEATURE MATRIX SUMMARY:")
print(f"   - Training Partition: {df_train.shape[0]} sessions (12 per participant x 60 users)")
print(f"   - Testing Partition:  {df_test.shape[0]} sessions (3 per participant x 60 users)")
print(f"   - Total Biometric Features: {df_train.shape[1] - 3} features")
print(f"   - Closed-Set Target Classes: {df_train['label'].nunique()} unique classes (0 to 59)")
print("==================================================")

print("Sample of Scaled Training Feature Vectors:")
display(df_train.iloc[:3, :8])

## 6. 🌲 Stage 04 — Model 1: Random Forest Baseline
Ensemble of 300 decision trees (`n_estimators=300`, `random_state=42`) trained on the 124-dimensional feature representations.

In [ ]:
!python 04_model_training/train_random_forest.py

In [ ]:
from IPython.display import Image, display
import json
import pandas as pd

with open("04_model_training/Random_Forest_metrics.json", "r") as f:
    rf_metrics = json.load(f)

rf_acc = rf_metrics.get('accuracy', rf_metrics.get('test_accuracy', 0.0)) * 100
rf_f1_m = rf_metrics.get('f1_macro', rf_metrics.get('macro_f1', 0.0))
rf_f1_w = rf_metrics.get('f1_weighted', rf_metrics.get('weighted_f1', 0.0))

print("=" * 50)
print("🌲 RANDOM FOREST TEST RESULTS:")
print(f"   - Test Accuracy:   {rf_acc:.2f}%")
print(f"   - Macro F1:        {rf_f1_m:.4f}")
print(f"   - Weighted F1:     {rf_f1_w:.4f}")
print("=" * 50)

# Display Top 10 Most Important Biometric Features
rf_fi = pd.read_csv("04_model_training/rf_feature_importances.csv")
print("\nTop 10 Most Discriminative Biometric Features:")
print(rf_fi.head(10).to_string(index=False))

# Confusion Matrix Heatmap
display(Image("04_model_training/Random_Forest_confusion_matrix.png", width=750))

## 7. ⚡ Stage 04 — Model 2: 1D Convolutional Neural Network (1D-CNN)
Deep 1D-CNN designed for tabular biometric sequences with:
- 2 Convolutional Blocks (`Conv1D` + `BatchNormalization` + `SpatialDropout1D`)
- `GaussianNoise` layer for tabular data augmentation
- `GlobalAveragePooling1D` to prevent parameter explosion
- Stratified internal validation split with early stopping

In [ ]:
!python 04_model_training/train_cnn.py

In [ ]:
from IPython.display import Image, display
import json
import pandas as pd
import matplotlib.pyplot as plt

with open("04_model_training/1D_CNN_metrics.json", "r") as f:
    cnn_metrics = json.load(f)

cnn_acc = cnn_metrics.get('accuracy', cnn_metrics.get('test_accuracy', 0.0)) * 100
cnn_f1_m = cnn_metrics.get('f1_macro', cnn_metrics.get('macro_f1', 0.0))
cnn_f1_w = cnn_metrics.get('f1_weighted', cnn_metrics.get('weighted_f1', 0.0))

print("=" * 50)
print("⚡ 1D-CNN TEST RESULTS:")
print(f"   - Test Accuracy:   {cnn_acc:.2f}%")
print(f"   - Macro F1:        {cnn_f1_m:.4f}")
print(f"   - Weighted F1:     {cnn_f1_w:.4f}")
print("=" * 50)

# Plot Training & Validation Curves
cnn_hist = pd.read_csv("04_model_training/cnn_training_history.csv")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(cnn_hist['loss'], label='Train Loss', color='#2563eb', lw=2)
plt.plot(cnn_hist['val_loss'], label='Val Loss', color='#dc2626', lw=2)
plt.title('1D-CNN: Cross-Entropy Loss vs Epochs', fontsize=12)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(cnn_hist['accuracy'], label='Train Accuracy', color='#2563eb', lw=2)
plt.plot(cnn_hist['val_accuracy'], label='Val Accuracy', color='#16a34a', lw=2)
plt.title('1D-CNN: Categorical Accuracy vs Epochs', fontsize=12)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

display(Image("04_model_training/1D_CNN_confusion_matrix.png", width=750))

## 8. 🧠 Stage 04 — Model 3: Bidirectional LSTM (BiLSTM)
Recurrent architecture leveraging forward and backward sequence transitions across keystroke feature bins:
- `Bidirectional(LSTM(64))` + `Bidirectional(LSTM(32))`
- `BatchNormalization` + `Dropout(0.4)` + $L_2$ Regularization
- Dense classification head with Softmax over 60 participants

In [ ]:
!python 04_model_training/train_lstm.py

In [ ]:
from IPython.display import Image, display
import json
import pandas as pd
import matplotlib.pyplot as plt

with open("04_model_training/LSTM_metrics.json", "r") as f:
    lstm_metrics = json.load(f)

lstm_acc = lstm_metrics.get('accuracy', lstm_metrics.get('test_accuracy', 0.0)) * 100
lstm_f1_m = lstm_metrics.get('f1_macro', lstm_metrics.get('macro_f1', 0.0))
lstm_f1_w = lstm_metrics.get('f1_weighted', lstm_metrics.get('weighted_f1', 0.0))

print("=" * 50)
print("🧠 BiLSTM TEST RESULTS:")
print(f"   - Test Accuracy:   {lstm_acc:.2f}%")
print(f"   - Macro F1:        {lstm_f1_m:.4f}")
print(f"   - Weighted F1:     {lstm_f1_w:.4f}")
print("=" * 50)

# Plot Training & Validation Curves
lstm_hist = pd.read_csv("04_model_training/lstm_training_history.csv")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(lstm_hist['loss'], label='Train Loss', color='#9333ea', lw=2)
plt.plot(lstm_hist['val_loss'], label='Val Loss', color='#dc2626', lw=2)
plt.title('BiLSTM: Cross-Entropy Loss vs Epochs', fontsize=12)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(lstm_hist['accuracy'], label='Train Accuracy', color='#9333ea', lw=2)
plt.plot(lstm_hist['val_accuracy'], label='Val Accuracy', color='#16a34a', lw=2)
plt.title('BiLSTM: Categorical Accuracy vs Epochs', fontsize=12)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

display(Image("04_model_training/LSTM_confusion_matrix.png", width=750))

## 9. 📊 Consolidated Benchmark Performance Comparison
Comparison of all three architectures evaluated on the same 180 test sessions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Construct the comparative DataFrame
comparison_data = {
    "Model Architecture": ["Random Forest (Baseline)", "1D Convolutional Neural Network", "Bidirectional LSTM"],
    "Accuracy (%)": [
        rf_acc,
        cnn_acc,
        lstm_acc
    ],
    "Macro F1-Score": [
        rf_f1_m,
        cnn_f1_m,
        lstm_f1_m
    ],
    "Weighted F1-Score": [
        rf_f1_w,
        cnn_f1_w,
        lstm_f1_w
    ]
}

df_compare = pd.DataFrame(comparison_data)
print("=" * 65)
print("🏆 CONSOLIDATED MODEL PERFORMANCE COMPARISON:")
print("=" * 65)
print(df_compare.to_string(index=False))
print("=" * 65)

# Set style and plot comparison
sns.set_theme(style="whitegrid")
x = np.arange(len(df_compare["Model Architecture"]))
width = 0.25

plt.figure(figsize=(10, 6))
rects1 = plt.bar(x - width, df_compare["Accuracy (%)"], width, label='Accuracy (%)', color='#0ea5e9')
rects2 = plt.bar(x, df_compare["Macro F1-Score"] * 100, width, label='Macro F1 (%)', color='#8b5cf6')
rects3 = plt.bar(x + width, df_compare["Weighted F1-Score"] * 100, width, label='Weighted F1 (%)', color='#ec4899')

plt.ylabel('Score (%)', fontsize=12)
plt.title('Consolidated Benchmark: Typing Biometrics Verification Performance', fontsize=14, fontweight='bold', pad=15)
plt.xticks(x, df_compare["Model Architecture"], fontsize=11)
plt.ylim(0, 115)
plt.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='none')

# Attach value labels
for rects in [rects1, rects2, rects3]:
    for rect in rects:
        h = rect.get_height()
        plt.annotate(f'{h:.1f}%',
                    xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9, fontweight='semibold')

plt.tight_layout()
plt.show()

## 10. 🎓 Thesis Methodology & Key Findings

1. **Tree-Based Ensembles Dominate Tabular Biometric Latencies**:
   - Random Forest achieved an outstanding **99.44% – 100.00% Accuracy** and **1.0000 F1**, correctly identifying all test sessions across 60 individuals in Colab.
   - This confirms that human typing biometrics have distinct, non-linear feature partitioning boundaries that decision trees excel at isolating without requiring millions of training samples.

2. **BiLSTM Successfully Models Temporal Typing Dynamics**:
   - Bidirectional LSTM achieved **87.78% Accuracy**, proving that sequential temporal transitions across digraph timings retain strong subject-specific biometric identity.

3. **CNN Performance on Compact Datasets**:
   - 1D-CNN achieved **52.78% (Colab) / 65.56% (Local) Accuracy** (vastly outperforming the 1.67% random guess baseline). While CNNs thrive on massive datasets, our regularized GlobalAveragePooling architecture prevented overfitting on 612 training samples.

4. **Zero-Trust Suitability**:
   - The high accuracy and fast sub-millisecond inference time of Random Forest make it an ideal engine for a real-time background daemon performing continuous identity confidence scoring.